# Classificação com o Dataset Iris

Neste notebook vamos aprender classificação na prática usando o **Iris dataset**.

## Problema
O Iris dataset contém medidas de flores de três espécies. Nosso objetivo é:

> "Dadas as medidas de uma flor (comprimento e largura de sépala e pétala), qual é a sua espécie?"

**Classes:** `setosa`, `versicolor`, `virginica`  
**Features:** `sepal length`, `sepal width`, `petal length`, `petal width`

## Algoritmos que vamos comparar
1. **Regressão Logística** — baseline clássico
2. **K-Nearest Neighbors (KNN)** — baseado em distância
3. **Árvore de Decisão** — baseado em regras interpretáveis
4. **Naive Bayes (Gaussiano)** — baseado no Teorema de Bayes


---
## 1. Importações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Bibliotecas carregadas com sucesso!')

---
## 2. Carregando e Explorando o Dataset

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df['species_id'] = iris.target

print('=== Iris Dataset ===')
print(f'Amostras: {df.shape[0]} | Features: {df.shape[1] - 2}')
print(f'\nClasses: {list(iris.target_names)}')
print(f'\nDistribuição das classes:')
print(df['species'].value_counts().to_string())
print('\nPrimeiras 5 linhas:')
df.drop(columns='species_id').head()

In [ ]:
print('=== Estatísticas por Espécie ===')
df.groupby('species')[iris.feature_names].mean().round(2)

---
## 3. Visualização Exploratória (EDA)

In [ ]:
# Pairplot — todas as combinações de features
g = sns.pairplot(
    df.drop(columns='species_id'),
    hue='species',
    diag_kind='hist',
    plot_kws={'alpha': 0.6, 's': 40},
    palette={'setosa': '#e74c3c', 'versicolor': '#2ecc71', 'virginica': '#3498db'}
)
g.figure.suptitle('Iris Dataset — Todas as Combinações de Features', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observação: petal length e petal width são as features mais discriminativas!')

In [ ]:
# Distribuição de cada feature por espécie
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribuição das Features por Espécie', fontsize=15, fontweight='bold')

colors = {'setosa': '#e74c3c', 'versicolor': '#2ecc71', 'virginica': '#3498db'}

for ax, feature in zip(axes.flatten(), iris.feature_names):
    for species, color in colors.items():
        subset = df[df['species'] == species][feature]
        ax.hist(subset, bins=15, alpha=0.6, label=species, color=color)
    ax.set_title(feature, fontsize=12)
    ax.set_xlabel('cm', fontsize=10)
    ax.set_ylabel('Frequência', fontsize=10)
    ax.legend(title='Espécie', fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Preparando os Dados

In [ ]:
# Features e target
X = iris.data   # todas as 4 features
y = iris.target # 0=setosa, 1=versicolor, 2=virginica

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    # stratify=y garante proporção igual das classes em treino e teste
)

print('=== Divisão dos Dados ===')
print(f'Total: {len(X)} amostras')
print(f'Treino: {len(X_train)} ({len(X_train)/len(X):.0%})')
print(f'Teste:  {len(X_test)} ({len(X_test)/len(X):.0%})')

print('\nClasses no treino:', np.bincount(y_train))
print('Classes no teste: ', np.bincount(y_test))

In [ ]:
# Normalização — importante para Regressão Logística e KNN
# (Árvore de Decisão e Naive Bayes não precisam, mas aplicamos a todos por consistência)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform no treino
X_test_scaled  = scaler.transform(X_test)        # apenas transform no teste

print('=== Normalização (StandardScaler) ===')
print('Antes — média das features no treino:', X_train.mean(axis=0).round(2))
print('Depois — média das features no treino:', X_train_scaled.mean(axis=0).round(4))
print('Depois — desvio padrão no treino:', X_train_scaled.std(axis=0).round(4))
print('\nObservação: StandardScaler → média=0, desvio=1 para cada feature')

---
## 5. Treinando os Modelos

### 5.1 Regressão Logística

In [ ]:
lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)

print('=== Regressão Logística ===')
print(f'Accuracy no teste: {acc_lr:.4f} ({acc_lr:.2%})')
print(f'\nCoeficientes aprendidos (uma linha por classe):')
coef_df = pd.DataFrame(
    lr.coef_,
    columns=iris.feature_names,
    index=iris.target_names
)
coef_df.round(3)

### 5.2 K-Nearest Neighbors (KNN)

In [ ]:
# Encontrar o melhor k via validação cruzada
k_scores = []
k_range = range(1, 21)

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

best_k = k_range[np.argmax(k_scores)]

plt.figure(figsize=(9, 4))
plt.plot(k_range, k_scores, 'o-', color='steelblue', lw=2, markersize=6)
plt.axvline(best_k, color='red', linestyle='--', label=f'Melhor k = {best_k}')
plt.xlabel('Valor de k', fontsize=12)
plt.ylabel('Accuracy (CV 5-fold)', fontsize=12)
plt.title('KNN — Escolha do Melhor k', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Melhor k: {best_k} | Accuracy média (CV): {max(k_scores):.4f}')

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)

print(f'=== KNN (k={best_k}) ===')
print(f'Accuracy no teste: {acc_knn:.4f} ({acc_knn:.2%})')

### 5.3 Árvore de Decisão

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
acc_dt = accuracy_score(y_test, y_pred_dt)

print('=== Árvore de Decisão (max_depth=3) ===')
print(f'Accuracy no teste: {acc_dt:.4f} ({acc_dt:.2%})')
print(f'\nImportância das features:')
for feat, imp in zip(iris.feature_names, dt.feature_importances_):
    bar = '█' * int(imp * 40)
    print(f'  {feat:30s} {bar} {imp:.4f}')

In [ ]:
# Visualizar a árvore de decisão
plt.figure(figsize=(16, 7))
plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    impurity=False
)
plt.title('Árvore de Decisão — Iris Dataset', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Cada nó interno: condição de divisão')
print('Cada folha: classe predita + amostras de cada classe')

### 5.4 Naive Bayes (Gaussiano)

O `GaussianNB` assume que cada feature segue uma **distribuição normal** dentro de cada classe.
No treino, ele estima `μ` (média) e `σ²` (variância) de cada feature por classe.
Na predição, usa o **Teorema de Bayes** para calcular a probabilidade posterior de cada classe.

```
P(classe | X) ∝ P(classe) · Π P(xᵢ | classe)
```

In [ ]:
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)

y_pred_nb = nb.predict(X_test_scaled)
acc_nb = accuracy_score(y_test, y_pred_nb)

print('=== Naive Bayes (Gaussiano) ===')
print(f'Accuracy no teste: {acc_nb:.4f} ({acc_nb:.2%})')

print('\nMédia (μ) aprendida por classe e feature:')
theta_df = pd.DataFrame(
    nb.theta_,
    columns=iris.feature_names,
    index=iris.target_names
)
print(theta_df.round(3))

print('\nVariância (σ²) aprendida por classe e feature:')
var_df = pd.DataFrame(
    nb.var_,
    columns=iris.feature_names,
    index=iris.target_names
)
print(var_df.round(3))

In [ ]:
# Visualizar as distribuições gaussianas aprendidas (para petal length)
from scipy.stats import norm

feature_idx = 2  # petal length (cm)
feature_name = iris.feature_names[feature_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Naive Bayes — Distribuições Gaussianas Aprendidas', fontsize=14, fontweight='bold')

cores_nb = {'setosa': '#e74c3c', 'versicolor': '#2ecc71', 'virginica': '#3498db'}

for feat_i, ax in zip([2, 3], axes):  # petal length e petal width
    feat_name = iris.feature_names[feat_i]
    x_range = np.linspace(X_train_scaled[:, feat_i].min() - 1,
                          X_train_scaled[:, feat_i].max() + 1, 300)

    for cls_i, (cls_name, color) in enumerate(cores_nb.items()):
        mu  = nb.theta_[cls_i, feat_i]
        std = np.sqrt(nb.var_[cls_i, feat_i])
        ax.plot(x_range, norm.pdf(x_range, mu, std),
                color=color, lw=2.5, label=f'{cls_name} (μ={mu:.2f}, σ={std:.2f})')
        ax.axvline(mu, color=color, lw=1, linestyle='--', alpha=0.5)

    ax.set_xlabel(f'{feat_name} (normalizado)', fontsize=11)
    ax.set_ylabel('Densidade de Probabilidade', fontsize=11)
    ax.set_title(feat_name, fontsize=12)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Cada curva = P(feature | classe) que o modelo aprendeu')
print('Na predição, essas curvas são multiplicadas para obter P(classe | X)')

---
## 6. Avaliação Detalhada — Métricas e Matrizes de Confusão

In [ ]:
# Matrizes de confusão — 2x2 para os 4 modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Matrizes de Confusão', fontsize=15, fontweight='bold')

modelos_avaliacao = [
    ('Regressão Logística', y_pred_lr, acc_lr),
    (f'KNN (k={best_k})', y_pred_knn, acc_knn),
    ('Árvore de Decisão', y_pred_dt, acc_dt),
    ('Naive Bayes', y_pred_nb, acc_nb),
]

for ax, (nome, y_pred, acc) in zip(axes.flatten(), modelos_avaliacao):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome}\nAccuracy: {acc:.2%}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predito', fontsize=10)
    ax.set_ylabel('Real', fontsize=10)

plt.tight_layout()
plt.show()
print('Diagonal principal = acertos | Fora da diagonal = erros')

In [ ]:
# Relatório completo por classe
print('=== Regressão Logística — Classification Report ===')
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names))

print(f'=== KNN (k={best_k}) — Classification Report ===')
print(classification_report(y_test, y_pred_knn, target_names=iris.target_names))

print('=== Árvore de Decisão — Classification Report ===')
print(classification_report(y_test, y_pred_dt, target_names=iris.target_names))

print('=== Naive Bayes — Classification Report ===')
print(classification_report(y_test, y_pred_nb, target_names=iris.target_names))

---
## 7. Comparação dos Modelos

In [ ]:
# Comparação com validação cruzada (5-fold)
resultados = {}

for nome, modelo in [
    ('Reg. Logística', LogisticRegression(max_iter=200, random_state=42)),
    (f'KNN (k={best_k})', KNeighborsClassifier(n_neighbors=best_k)),
    ('Árvore de Decisão', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Naive Bayes', GaussianNB()),
]:
    scores = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring='accuracy')
    resultados[nome] = scores

print('=== Comparação via Cross-Validation (5-fold no treino) ===')
print(f'\n  {"Modelo":20s} | {"Média":>8} | {"Desvio":>8} | {"Min":>7} | {"Max":>7}')
print(f'  {"-"*20}-+-{"-"*8}-+-{"-"*8}-+-{"-"*7}-+-{"-"*7}')
for nome, scores in resultados.items():
    print(f'  {nome:20s} | {scores.mean():.4f}   | {scores.std():.4f}   | {scores.min():.4f}  | {scores.max():.4f}')

print(f'\n  Accuracy no conjunto de TESTE:')
print(f'  Regressão Logística: {acc_lr:.4f}')
print(f'  KNN (k={best_k}):         {acc_knn:.4f}')
print(f'  Árvore de Decisão:   {acc_dt:.4f}')
print(f'  Naive Bayes:         {acc_nb:.4f}')

In [ ]:
# Gráfico de comparação
fig, ax = plt.subplots(figsize=(11, 5))

nomes = list(resultados.keys())
medias = [resultados[n].mean() for n in nomes]
desvios = [resultados[n].std() for n in nomes]
cores = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

bars = ax.bar(nomes, medias, yerr=desvios, capsize=8,
              color=cores, alpha=0.8, edgecolor='white', linewidth=1.5)

for bar, media in zip(bars, medias):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{media:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0.85, 1.05)
ax.set_ylabel('Accuracy (CV 5-fold)', fontsize=12)
ax.set_title('Comparação de Modelos — Iris Dataset\n(barra de erro = desvio padrão do CV)', fontsize=13, fontweight='bold')
ax.axhline(1.0, color='gray', linestyle=':', lw=1)

plt.tight_layout()
plt.show()

---
## 8. Fronteiras de Decisão (Visualização)

Para visualizar em 2D, usamos apenas as 2 features mais importantes: `petal length` e `petal width`.

In [ ]:
# Usar apenas petal length e petal width para plot 2D
X_2d = iris.data[:, 2:4]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.2, random_state=42, stratify=y
)

scaler_2d = StandardScaler()
X_train_2d_sc = scaler_2d.fit_transform(X_train_2d)
X_test_2d_sc  = scaler_2d.transform(X_test_2d)

modelos_2d = [
    ('Regressão Logística', LogisticRegression(max_iter=200, random_state=42)),
    (f'KNN (k={best_k})', KNeighborsClassifier(n_neighbors=best_k)),
    ('Árvore de Decisão', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Naive Bayes', GaussianNB()),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Fronteiras de Decisão — Petal Length × Petal Width', fontsize=15, fontweight='bold')

scatter_colors = ['#e74c3c', '#2ecc71', '#3498db']

x_min, x_max = X_train_2d_sc[:, 0].min() - 0.5, X_train_2d_sc[:, 0].max() + 0.5
y_min, y_max = X_train_2d_sc[:, 1].min() - 0.5, X_train_2d_sc[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

for ax, (nome, modelo) in zip(axes.flatten(), modelos_2d):
    modelo.fit(X_train_2d_sc, y_train_2d)
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    acc = accuracy_score(y_test_2d, modelo.predict(X_test_2d_sc))

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)

    for cls, color in zip([0, 1, 2], scatter_colors):
        mask = y_train_2d == cls
        ax.scatter(X_train_2d_sc[mask, 0], X_train_2d_sc[mask, 1],
                   color=color, alpha=0.7, s=40, label=iris.target_names[cls], zorder=3)

    ax.set_xlabel('Petal Length (norm.)', fontsize=10)
    ax.set_ylabel('Petal Width (norm.)', fontsize=10)
    ax.set_title(f'{nome}\nAccuracy (2D): {acc:.2%}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Observe a fronteira curva do Naive Bayes — resultado das distribuições gaussianas!')

---
## 9. Fazendo Predições com Novos Dados

In [ ]:
# Novas flores para classificar
# [sepal_length, sepal_width, petal_length, petal_width]
novas_flores = np.array([
    [5.1, 3.5, 1.4, 0.2],  # características típicas de setosa
    [6.2, 2.9, 4.3, 1.3],  # características típicas de versicolor
    [7.2, 3.0, 5.8, 1.6],  # características típicas de virginica
    [5.5, 2.6, 4.0, 1.2],  # caso ambíguo
])

novas_flores_scaled = scaler.transform(novas_flores)

pred_lr  = lr.predict(novas_flores_scaled)
pred_knn = knn.predict(novas_flores_scaled)
pred_dt  = dt.predict(novas_flores_scaled)
pred_nb  = nb.predict(novas_flores_scaled)

prob_lr = lr.predict_proba(novas_flores_scaled)
prob_nb = nb.predict_proba(novas_flores_scaled)

print('=== Predições para Novas Flores ===')
print(f'\n  {"Flor":>4} | {"Reg. Logística":>16} | {"KNN":>10} | {"Árv. Decisão":>14} | {"Naive Bayes":>12}')
print(f'  {"-"*4}-+-{"-"*16}-+-{"-"*10}-+-{"-"*14}-+-{"-"*12}')
for i, (p_lr, p_knn, p_dt, p_nb) in enumerate(zip(pred_lr, pred_knn, pred_dt, pred_nb)):
    print(f'  {i+1:>4} | {iris.target_names[p_lr]:>16} | {iris.target_names[p_knn]:>10} | {iris.target_names[p_dt]:>14} | {iris.target_names[p_nb]:>12}')

In [ ]:
# Comparar probabilidades: Regressão Logística vs Naive Bayes
fig, axes = plt.subplots(2, len(novas_flores), figsize=(14, 7))
fig.suptitle('Probabilidades por Classe — Logística vs Naive Bayes', fontsize=13, fontweight='bold')

cores = ['#e74c3c', '#2ecc71', '#3498db']

for col, (probs_lr, probs_nb) in enumerate(zip(prob_lr, prob_nb)):
    for row, (probs, modelo_nome, preds) in enumerate([
        (probs_lr, 'Reg. Logística', pred_lr),
        (probs_nb, 'Naive Bayes', pred_nb),
    ]):
        ax = axes[row][col]
        bars = ax.bar(iris.target_names, probs, color=cores, alpha=0.8, edgecolor='white')
        ax.set_ylim(0, 1.15)
        ax.set_title(f'Flor {col+1} — {modelo_nome}\nPredição: {iris.target_names[preds[col]]}', fontsize=10)
        ax.set_xticklabels(iris.target_names, rotation=15, fontsize=9)
        for bar, prob in zip(bars, probs):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{prob:.2%}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

---
## 10. Naive Bayes — Verificação Passo a Passo

Vamos calcular manualmente a predição do Naive Bayes para entender o que acontece internamente.

In [ ]:
from scipy.stats import norm

idx = 0
x_amostra        = X_test[idx]
x_amostra_scaled = X_test_scaled[idx]
y_real           = y_test[idx]

print('=== Naive Bayes — Cálculo Manual ===')
print(f'\nAmostra: {x_amostra} → classe real: {iris.target_names[y_real]}')
print(f'Amostra normalizada: {x_amostra_scaled.round(4)}')

print('\n--- Passo 1: Prior P(classe) ---')
for cls_i, cls_name in enumerate(iris.target_names):
    prior = nb.class_prior_[cls_i]
    print(f'  P({cls_name}) = {prior:.4f}')

print('\n--- Passo 2: Verossimilhança P(X | classe) para cada feature ---')
log_likelihoods = np.zeros(3)
for cls_i, cls_name in enumerate(iris.target_names):
    print(f'\n  Classe: {cls_name}')
    log_lk = 0
    for feat_i, feat_name in enumerate(iris.feature_names):
        mu  = nb.theta_[cls_i, feat_i]
        std = np.sqrt(nb.var_[cls_i, feat_i])
        p   = norm.pdf(x_amostra_scaled[feat_i], mu, std)
        log_lk += np.log(p + 1e-300)
        print(f'    P({feat_name[:15]:15s} = {x_amostra_scaled[feat_i]:+.3f} | {cls_name}) = {p:.6f}  (μ={mu:.3f}, σ={std:.3f})')
    log_likelihoods[cls_i] = log_lk

print('\n--- Passo 3: Posterior (log escala) ---')
log_posteriors = log_likelihoods + np.log(nb.class_prior_)
for cls_i, cls_name in enumerate(iris.target_names):
    print(f'  log P({cls_name} | X) ∝ {log_posteriors[cls_i]:.4f}')

pred_manual = iris.target_names[np.argmax(log_posteriors)]
pred_modelo = iris.target_names[nb.predict([x_amostra_scaled])[0]]

print(f'\n  Predição manual:  {pred_manual}')
print(f'  Predição modelo:  {pred_modelo}')
print(f'  Classe real:      {iris.target_names[y_real]}')
print(f'  Manual == Modelo? {pred_manual == pred_modelo}')

---
## 11. Resumo Final

In [ ]:
print('=' * 62)
print('           RESUMO — CLASSIFICAÇÃO IRIS')
print('=' * 62)
print()
print('  PROBLEMA')
print('  Classificar espécie de flor (3 classes) a partir de 4 features')
print()
print('  DADOS')
print('  Total: 150 | Treino: 120 | Teste: 30 | Classes balanceadas')
print()
print('  PRÉ-PROCESSAMENTO')
print('  StandardScaler → média=0, desvio=1')
print()
print('  RESULTADOS (conjunto de teste)')
print(f'  Regressão Logística : {acc_lr:.4f} ({acc_lr:.2%})')
print(f'  KNN (k={best_k})          : {acc_knn:.4f} ({acc_knn:.2%})')
print(f'  Árvore de Decisão   : {acc_dt:.4f} ({acc_dt:.2%})')
print(f'  Naive Bayes         : {acc_nb:.4f} ({acc_nb:.2%})')
print()
print('  CARACTERÍSTICAS DOS ALGORITMOS')
print('  Reg. Logística → fronteira linear, probabilidades calibradas')
print('  KNN            → fronteira irregular, sensível à escala')
print('  Árvore Decisão → fronteiras em ângulo reto, interpretável')
print('  Naive Bayes    → fronteira curva, rápido, assume independência')
print()
print('  OBSERVAÇÕES')
print('  - setosa é perfeitamente separável por todos os modelos')
print('  - versicolor e virginica têm alguma sobreposição')
print('  - petal_length e petal_width são as features mais discriminativas')
print('=' * 62)

---
## Próximos Passos

1. **Random Forest** — ensemble de árvores de decisão
2. **SVM** — Support Vector Machine, ótimo para margens máximas
3. **Grid Search** — otimização automática de hiperparâmetros
4. **Pipeline** — encadear scaler + modelo de forma limpa

```python
# Exemplo: Pipeline com scaler + modelo
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GaussianNB())
])
pipe.fit(X_train, y_train)
pipe.predict(X_test)
```